# 01 - Data Exploration (EDA)

Goal: understand the Booking.com Multi-Destination Trips (MDT) dataset well enough to justify our modelling and evaluation choices in the report. Specifically we check:

1. Trip-length distribution (and whether the **test** set really has a minimum of 4 stops - this justifies our train filter).
2. How many users vs trips vs cities, and how sparse per-user history is (this motivates the FPMC **persona** instead of raw `user_id`).
3. Seasonality - does the month of travel relate to destination.
4. Why **leave-last-out** is the right evaluation for this data.

In [ ]:
import os, sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

def find_root(start=None):
    d = os.path.abspath(start or os.getcwd())
    for _ in range(6):
        if os.path.exists(os.path.join(d, 'src', 'preprocessing.py')):
            return d
        d = os.path.dirname(d)
    return os.path.abspath(os.path.join(os.getcwd(), '..'))

ROOT = find_root(); sys.path.insert(0, ROOT)
DATA_DIR = os.path.join(ROOT, 'data')
# Colab: uncomment to fetch CSVs -> data/
# !python {os.path.join(ROOT,'scripts','download_data.py')} --out {DATA_DIR}
from src.preprocessing import load_raw
train, test, gt = load_raw(DATA_DIR)
print('train rows:', len(train), '| test rows:', len(test), '| ground_truth rows:', len(gt))
train.head()

In [ ]:
# --- Trip-length distribution (train vs test) ---
train_len = train.groupby('utrip_id').size()
test_len = test.groupby('utrip_id').size()
print('TRAIN trip length: min=%d  median=%.1f  max=%d' % (train_len.min(), train_len.median(), train_len.max()))
print('TEST  trip length: min=%d  median=%.1f  max=%d' % (test_len.min(), test_len.median(), test_len.max()))
print('\n(test includes the masked final row, so its true trip length is one more)')

fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
train_len.clip(upper=12).hist(bins=range(1, 14), ax=ax[0]); ax[0].set_title('TRAIN trip length'); ax[0].set_xlabel('stops')
test_len.clip(upper=12).hist(bins=range(1, 14), ax=ax[1]); ax[1].set_title('TEST trip length'); ax[1].set_xlabel('stops')
plt.tight_layout(); plt.show()

In [ ]:
# --- Users, trips, cities, and per-user sparsity ---
n_trips = train['utrip_id'].nunique()
n_users = train['user_id'].nunique()
n_cities = train['city_id'].nunique()
trips_per_user = train.groupby('user_id')['utrip_id'].nunique()
print('trips :', n_trips)
print('users :', n_users)
print('cities:', n_cities)
print('\ntrips per user: mean=%.2f  median=%d  max=%d' % (trips_per_user.mean(), trips_per_user.median(), trips_per_user.max()))
print('%% of users with exactly 1 trip: %.1f%%' % (100 * (trips_per_user == 1).mean()))

> **Report note.** If almost all users have a single trip, a per-`user_id` latent factor cannot generalise to the test trips (every test user is effectively cold-start). This is exactly why FPMC is run with a coarse **persona** = (booker_country, device_class) that recurs across many trips, and why we expect raw-`user_id` FPMC to underperform it.

In [ ]:
# --- Seasonality: does month relate to which cities are popular? ---
train['checkin'] = pd.to_datetime(train['checkin'], errors='coerce')
train['month'] = train['checkin'].dt.month
by_month = train.groupby('month')['city_id'].agg(['count', 'nunique'])
print(by_month)
by_month['count'].plot(kind='bar', figsize=(8, 3), title='bookings per month'); plt.tight_layout(); plt.show()
# how much does the top-10 city set change month to month?
top_by_month = {m: set(g['city_id'].value_counts().head(10).index) for m, g in train.groupby('month')}
months = sorted(top_by_month)
overlap = np.mean([len(top_by_month[months[i]] & top_by_month[months[i+1]]) / 10 for i in range(len(months)-1)])
print('\nmean top-10 city overlap between consecutive months: %.2f (1.0 = no seasonal shift)' % overlap)

In [ ]:
# --- Context field distributions (feed SASRec side-features / FPMC persona) ---
for col in ['device_class', 'booker_country', 'affiliate_id']:
    vc = train[col].value_counts()
    print(f'{col}: {train[col].nunique()} distinct; top 3 = {list(vc.head(3).index)}')

## Takeaways for the report

- **Leave-last-out fits naturally.** Each trip is a short ordered sequence; hiding the final stop mirrors the real product task (recommend the next destination given the trip so far), and the dataset ships this split for us (masked test row + `ground_truth.csv`).
- **Minimum length justifies the train filter.** (Fill in the observed test minimum from the length cell above.)
- **Per-user history is sparse**, motivating the persona-based FPMC variant over raw `user_id`.
- **Seasonality / context** (month, device, country) is worth encoding as side-features for SASRec and as the persona for FPMC; the overlap statistic above quantifies how much the destination mix shifts across the year.